> **Experiment-source record.** This notebook is retained to document the code path used for the released run. Outputs and credentials have been removed. The canonical numerical reproduction starts from the persisted CSV/JSON records via `00_reproduce_released analysis_analysis.ipynb`.


# Axis-2 feature-selection audit

Standalone. Nothing from the other notebooks is needed.

**The objection.** `ids_llm_nebius_v8.ipynb` computes
`FEATURES = select_top_features(data, ...)` once, on the pooled labelled frame, *before* the
per-seed ICL/test split. Features are therefore chosen with help from the labels of rows that
later serve as test rows. Because the same features feed both the LLMs and the Random Forest
anchor, the audit tests whether the paired conclusion survives while absolute values may be
inflated — and asks for that to be shown rather than assumed.

**Design.** Three selectors on identical ICL and test rows:

| regime | mutual information fitted on | status |
|---|---|---|
| `legacy_pooled` | the whole labelled frame | reproduces what was reported |
| `development_only` | a disjoint development partition | leakage-free |
| `icl_only` | the 5-shot rows alone | strictest |

**Two stages, and stage 2 may not be needed.**
Stage 1 runs the Random Forest anchor only: free, a few minutes. If the absolute numbers barely
move, there is nothing to correct and the answer is one paragraph plus a table. Stage 2 runs one
LLM to show the gap is unchanged, and is only worth its cost if stage 1 says the numbers moved.

The notebook also measures cross-seed test-row reuse, which settles a separate claim: the
earlier protocol assumed that no test rows are shared across seeds, while each seed in fact draws an
independent permutation of the same per-class pool.

In [ ]:
!pip -q install kagglehub pyarrow scikit-learn openai 2>/dev/null

import itertools, json, os, re, sys, time, zipfile
from pathlib import Path
from datetime import datetime, timezone
import numpy as np, pandas as pd

try:
    from google.colab import userdata
    for k in ("KAGGLE_USERNAME", "KAGGLE_KEY", "NEBIUS_API_KEY"):
        try:
            v = userdata.get(k)
            if v: os.environ[k] = v
        except Exception:
            pass
except Exception:
    pass

OUT = Path("/content/axis2_mi"); OUT.mkdir(parents=True, exist_ok=True)
# Uncomment to survive a session timeout:
# from google.colab import drive; drive.mount("/content/drive")
# OUT = Path("/content/drive/MyDrive/axis2_mi"); OUT.mkdir(parents=True, exist_ok=True)

pd.set_option("display.width", 220)
print("kaggle creds:", bool(os.environ.get("KAGGLE_KEY")),
      "| nebius key:", bool(os.environ.get("NEBIUS_API_KEY")))
print("output:", OUT)

In [ ]:
# Protocol constants — these mirror ids_llm_nebius_v8.ipynb so the audit is
# measured on the same footing as the reported panel. Only WHERE mutual
# information is fitted differs between regimes.
TOP_K            = 12        # v8: TOP_K_FEATURES
N_ICL_PER_CLASS  = 5         # v8: N_ICL_PER_CLASS
N_TEST_PER_CLASS = 50        # v8: N_TEST_PER_CLASS at SCALE="paper"
N_DEV_PER_CLASS  = 500       # new: the disjoint partition a clean selector needs
SUBSAMPLE_PER_CLASS = 50_000 # v8: tractability cap before mutual information
RF_TREES         = 200       # v8 anchor: RandomForestClassifier(n_estimators=200)
SEEDS            = list(range(10))
MI_SAMPLE        = 50_000    # v8: select_top_features(sample=50_000)

RUN_STAGE1_RF  = True        # free, minutes
RUN_STAGE2_LLM = False       # only if stage 1 shows the numbers moving
LLM_MODEL      = os.environ.get("AXIS2_MODEL_ID", "Qwen/Qwen3-235B-A22B-Instruct-2507")
LLM_SEEDS      = list(range(10))
LLM_N_VOTE     = 1           # temperature 0 makes a majority vote near-redundant
LLM_MAX_TOKENS = 512

print(f"K={TOP_K} shots={N_ICL_PER_CLASS} test/class={N_TEST_PER_CLASS} seeds={len(SEEDS)}")
print(f"stage 1 (RF): {RUN_STAGE1_RF}   stage 2 (LLM): {RUN_STAGE2_LLM}  model={LLM_MODEL}")

## 1. Data

`dhoogla/cicids2017` with the grouped 33→9 taxonomy and a 50,000-row-per-class cap, exactly as
the reported Axis-2 configuration prepares it.

In [ ]:
import kagglehub

CIC_MAP = {'BENIGN':'BENIGN','Benign':'BENIGN','DDoS':'DDoS',
           'DoS Hulk':'DoS','DoS GoldenEye':'DoS','DoS slowloris':'DoS','DoS Slowhttptest':'DoS',
           'PortScan':'Port Scan','Portscan':'Port Scan',
           'FTP-Patator':'Brute Force','SSH-Patator':'Brute Force','Bruteforce':'Brute Force',
           'Bot':'Bot','Infiltration':'Infiltration','Heartbleed':'Heartbleed'}
def group(v):
    s = re.sub(r"[^\x00-\x7F]+", " ", str(v)).replace("-", " ")
    s = re.sub(r"\s+", " ", s).strip()
    if "web attack" in s.lower(): return "Web Attack"
    return CIC_MAP.get(str(v).strip(), CIC_MAP.get(s, s))

CACHE = OUT/"axis2_table.parquet"
if CACHE.exists():
    data = pd.read_parquet(CACHE)
    print(f"loaded cached table: {len(data):,} rows")
else:
    folder = Path(kagglehub.dataset_download("dhoogla/cicids2017"))
    files = sorted(list(folder.rglob("*.parquet")) + list(folder.rglob("*.csv")))
    raw = pd.concat([(pd.read_parquet(f) if f.suffix==".parquet"
                      else pd.read_csv(f, low_memory=True)) for f in files], ignore_index=True)
    raw.columns = [c.strip().replace(" ", "_") for c in raw.columns]
    lab = "ClassLabel" if "ClassLabel" in raw.columns else "Label"
    raw["Label"] = raw[lab].map(group)
    num = raw.select_dtypes(include=[np.number]).columns.tolist()
    data = raw[num+["Label"]].replace([np.inf,-np.inf], np.nan).dropna().reset_index(drop=True)
    if data["Label"].value_counts().max() > SUBSAMPLE_PER_CLASS:
        data = (data.groupby("Label", group_keys=False)
                    .apply(lambda g: g.sample(min(len(g), SUBSAMPLE_PER_CLASS), random_state=0))
                    .reset_index(drop=True))
    data.to_parquet(CACHE, index=False)
    print(f"prepared and cached: {len(data):,} rows")

data["__rid__"] = np.arange(len(data))          # stable row identity for the overlap audit
LABELS = sorted(data.Label.astype(str).unique())
FEATURE_COLS = [c for c in data.columns if c not in ("Label","__rid__")]
print(f"{len(LABELS)} classes, {len(FEATURE_COLS)} numeric features")
print(data.Label.value_counts().to_string())

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import mutual_info_classif
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score

def mi_select(df, k=TOP_K, seed=0, sample=None):
    """Top-k features by mutual information. `sample` mirrors v8's tractability cap."""
    s = df.sample(min(len(df), sample), random_state=seed) if (sample and len(df) > sample) else df
    x = s[[c for c in FEATURE_COLS if c in s.columns]]
    y = LabelEncoder().fit_transform(s["Label"].astype(str))
    return list(x.columns[np.argsort(mutual_info_classif(x, y, random_state=seed))[::-1][:k]])

def split3(df, seed):
    """Disjoint development / in-context / test partitions, class-balanced.

    The development partition is what makes a leakage-free selector possible:
    mutual information can be fitted there without touching a test label.
    """
    rng = np.random.default_rng(seed)
    dev, icl, test, audit = [], [], [], []
    for cls, p in df.groupby("Label", sort=True):
        p = p.iloc[rng.permutation(len(p))]
        nd = min(N_DEV_PER_CLASS, max(0, len(p)-N_ICL_PER_CLASS-N_TEST_PER_CLASS))
        rest = len(p)-nd
        ni = min(N_ICL_PER_CLASS, rest); nt = min(N_TEST_PER_CLASS, max(0, rest-ni))
        dev.append(p.iloc[:nd]); icl.append(p.iloc[nd:nd+ni]); test.append(p.iloc[nd+ni:nd+ni+nt])
        audit.append({"class":cls,"available":len(p),"n_dev":nd,"n_icl":ni,"n_test":nt})
    cat = lambda ps: pd.concat([x for x in ps if len(x)], ignore_index=True)
    return cat(dev), cat(icl), cat(test), pd.DataFrame(audit)

def rf_score(icl, test, feats, seed):
    rf = RandomForestClassifier(n_estimators=RF_TREES, random_state=seed, n_jobs=-1)
    rf.fit(icl[feats].to_numpy(), icl["Label"].astype(str).to_numpy())
    p = rf.predict(test[feats].to_numpy())
    y = test["Label"].astype(str)
    return accuracy_score(y, p), f1_score(y, p, average="macro", zero_division=0)

def signflip(d):
    d = np.asarray(d, float); o = abs(d.mean())
    return float(np.mean([abs(np.mean(d*np.array(s))) >= o-1e-15
                          for s in itertools.product((-1,1), repeat=len(d))])) if len(d)<=20 else np.nan
def bci(d, n=10000, conf=0.95):
    d = np.asarray(d, float); b = np.random.default_rng(0).choice(d,(n,len(d))).mean(1)
    return tuple(np.percentile(b, [100*(1-conf)/2, 100*(1+conf)/2]))
def save(df, name):
    df.to_csv(OUT/name, index=False); print(f"[saved] {name} ({len(df)} rows)"); return df
print("protocol ready")

## 2. Stage 1 — the Random Forest anchor under three selectors

`legacy_pooled` is fitted once with seed 0 on a 50,000-row sample of the whole frame, exactly as
the reported protocol does, and then reused for every seed. The other two are refitted per seed
on partitions that exclude the test rows.

In [ ]:
if RUN_STAGE1_RF:
    t0 = time.time()
    legacy = mi_select(data, TOP_K, seed=0, sample=MI_SAMPLE)   # the reported protocol
    print(f"legacy_pooled features: {legacy}\n")

    rows, jac, test_sets, feat_log = [], [], {}, []
    for seed in SEEDS:
        dev, icl, test, audit = split3(data, seed)
        regimes = {"legacy_pooled":    legacy,
                   "development_only": mi_select(dev, TOP_K, seed),
                   "icl_only":         mi_select(icl, TOP_K, seed)}
        test_sets[seed] = set(test["__rid__"].astype(int))
        for a, b in itertools.combinations(regimes, 2):
            sa, sb = set(regimes[a]), set(regimes[b])
            jac.append({"seed":seed,"a":a,"b":b,"shared":len(sa&sb),
                        "jaccard":len(sa&sb)/max(1,len(sa|sb))})
        for reg, feats in regimes.items():
            acc, mf1 = rf_score(icl, test, feats, seed)
            rows.append({"seed":seed,"regime":reg,"model":"random_forest",
                         "accuracy":acc,"macro_f1":mf1,
                         "n_dev":len(dev),"n_icl":len(icl),"n_test":len(test),
                         "min_test_per_class":int(audit.n_test.min())})
            feat_log.append({"seed":seed,"regime":reg,"features":json.dumps(feats)})
        print(f"  seed {seed}: " + "  ".join(
            f"{r['regime']}={r['accuracy']:.4f}" for r in rows[-3:]))

    RF = save(pd.DataFrame(rows), "stage1_rf_regimes.csv")
    save(pd.DataFrame(feat_log), "stage1_selected_features.csv")
    JA = save(pd.DataFrame(jac), "stage1_selector_jaccard.csv")

    ov = [{"seed_a":a,"seed_b":b,
           "reuse_pct":100*len(test_sets[a]&test_sets[b])/max(1,len(test_sets[a])),
           "jaccard":len(test_sets[a]&test_sets[b])/max(1,len(test_sets[a]|test_sets[b]))}
          for a,b in itertools.combinations(sorted(test_sets),2)]
    OV = save(pd.DataFrame(ov), "stage1_test_overlap.csv")
    print(f"\nstage 1 finished in {(time.time()-t0)/60:.1f} min")
else:
    RF = pd.read_csv(OUT/"stage1_rf_regimes.csv") if (OUT/"stage1_rf_regimes.csv").exists() else pd.DataFrame()
    JA = pd.read_csv(OUT/"stage1_selector_jaccard.csv") if (OUT/"stage1_selector_jaccard.csv").exists() else pd.DataFrame()
    OV = pd.read_csv(OUT/"stage1_test_overlap.csv") if (OUT/"stage1_test_overlap.csv").exists() else pd.DataFrame()
    print("stage 1 disabled; loaded any previous results")

In [ ]:
if not RF.empty:
    print("=== anchor accuracy and macro-F1 by selector, 10 seeds ===")
    print(RF.groupby("regime")[["accuracy","macro_f1"]].agg(["mean","std"]).round(4).to_string())

    print("\n=== paired difference from the reported (leaky) protocol ===")
    base = RF[RF.regime=="legacy_pooled"].set_index("seed")
    rows=[]
    for reg in ("development_only","icl_only"):
        g = RF[RF.regime==reg].set_index("seed")
        k = sorted(set(g.index)&set(base.index))
        for met in ("accuracy","macro_f1"):
            d = g.loc[k,met].to_numpy()-base.loc[k,met].to_numpy(); lo,hi = bci(d)
            rows.append({"regime":reg,"metric":met,"n":len(d),"mean_delta":d.mean(),
                         "ci_lo":lo,"ci_hi":hi,"p":signflip(d),
                         "leaky_higher":int((d<0).sum())})
    LK = save(pd.DataFrame(rows), "stage1_leakage_effect.csv")
    print(LK.to_string(index=False, float_format=lambda v:f"{v: .4f}"))

    worst = LK[LK.metric=="accuracy"].mean_delta.abs().max()
    print(f"\nlargest absolute accuracy shift under leakage-free selection: {worst:.4f}")
    if worst < 0.01:
        print("VERDICT: the pooled selector inflates the anchor by less than one accuracy point.")
        print("Stage 2 is NOT needed: report stage 1, state that MI was fitted on the pooled")
        print("frame, and give this table as the correction.")
    else:
        print("VERDICT: absolute numbers move materially. Run stage 2 on one LLM to show the")
        print("LLM-minus-anchor gap is unchanged; that is what the parity claim rests on.")

    print("\n=== selector agreement (Jaccard between regimes) ===")
    print(JA.groupby(["a","b"]).jaccard.agg(["mean","min","max"]).round(3).to_string())
    print("\n=== cross-seed test-row reuse ===")
    print(OV[["reuse_pct","jaccard"]].agg(["mean","max"]).round(2).to_string())
    print("The earlier protocol assumed no test rows are shared across seeds. Replace that with the")
    print("measured reuse above and describe the seeds as repeated resamples.")

## 3. Stage 2 — one LLM under the same three selectors *(gated)*

Identical ICL rows, identical test rows and identical columns as the anchor, so the
LLM-minus-anchor gap is measured on matched inputs under each selector. Use whichever model the
released panel identifies as strongest; that is the one the parity claim is about.

Roughly 13,500 calls at these settings. Raw generations are persisted, since they are what makes
a hosted-endpoint result checkable later.

In [ ]:
LLM = pd.DataFrame()
if RUN_STAGE2_LLM:
    from openai import OpenAI
    client = OpenAI(base_url="https://api.studio.nebius.com/v1/",
                    api_key=os.environ["NEBIUS_API_KEY"])

    def parse_label(text):
        if not text: return None
        t = re.sub(r"<(think|thinking|reasoning)>.*?</\1>", " ", text, flags=re.S|re.I)
        t = re.sub(r"^.*</(think|thinking|reasoning)>", " ", t, flags=re.S|re.I)
        for line in [l.strip(" \t\"'`*.-:") for l in t.strip().splitlines()[::-1] if l.strip()]:
            for lab in sorted(LABELS, key=len, reverse=True):
                if line.lower() == lab.lower(): return lab
            for lab in sorted(LABELS, key=len, reverse=True):
                if lab.lower() in line.lower(): return lab
        return None

    def ask(icl, row, feats):
        shots = "\n".join(
            json.dumps({**{f: round(float(r[f]),4) for f in feats}, "label": str(r["Label"])})
            for _, r in icl.iterrows())
        system = ("You are a network intrusion detection classifier. "
                  f"Choose exactly one label from this list: {', '.join(LABELS)}. "
                  "Respond with the label only, on its own line, with no explanation.")
        user = f"{shots}\n" + json.dumps({f: round(float(row[f]),4) for f in feats})
        r = client.chat.completions.create(model=LLM_MODEL, max_tokens=LLM_MAX_TOKENS,
                                           temperature=0.0, timeout=300,
                                           messages=[{"role":"system","content":system},
                                                     {"role":"user","content":user}])
        u = getattr(r, "usage", None)
        return (r.choices[0].message.content or "",
                getattr(u,"prompt_tokens",0) or 0, getattr(u,"completion_tokens",0) or 0)

    legacy = mi_select(data, TOP_K, seed=0, sample=MI_SAMPLE)
    RESULTS_CSV = OUT/"stage2_llm_regimes.csv"
    done = set()
    if RESULTS_CSV.exists():
        prev = pd.read_csv(RESULTS_CSV); done = set(zip(prev.seed, prev.regime))
        print(f"resuming: {len(done)} (seed, regime) pairs complete")

    for seed in LLM_SEEDS:
        dev, icl, test, _ = split3(data, seed)
        regimes = {"legacy_pooled":legacy,
                   "development_only":mi_select(dev,TOP_K,seed),
                   "icl_only":mi_select(icl,TOP_K,seed)}
        y = test["Label"].astype(str)
        for reg, feats in regimes.items():
            if (seed, reg) in done: continue
            t0=time.time(); preds, raw, lat, err, ti, to = [], [], [], 0, 0, 0
            for _, row in test.iterrows():
                try:
                    s=time.time(); txt,a,b = ask(icl,row,feats); lat.append(time.time()-s)
                    ti+=a; to+=b; raw.append(txt); preds.append(parse_label(txt))
                except Exception as e:
                    err+=1; raw.append(f"__error__ {type(e).__name__}"); preds.append(None)
            yp = pd.Series(preds).fillna("__unparsed__").astype(str)
            pd.DataFrame([{ "seed":seed,"regime":reg,"model":LLM_MODEL,
                "accuracy":accuracy_score(y,yp),
                "macro_f1":f1_score(y,yp,average="macro",zero_division=0),
                "parse_rate":float(np.mean([p is not None for p in preds])),
                "errors":err,"prompt_tokens":ti,"completion_tokens":to,
                "latency_median_s":float(np.median(lat)) if lat else np.nan,
                "wall_min":(time.time()-t0)/60}]).to_csv(
                    RESULTS_CSV, mode="a", index=False, header=not RESULTS_CSV.exists())
            with (OUT/"stage2_raw_generations.jsonl").open("a") as fh:
                fh.write(json.dumps({"seed":seed,"regime":reg,"features":feats,
                                     "y_true":y.tolist(),"y_pred":preds,"raw":raw})+"\n")
            done.add((seed,reg))
            r = pd.read_csv(RESULTS_CSV).iloc[-1]
            print(f"  seed {seed} {reg:18} acc={r.accuracy:.4f} parse={r.parse_rate:.2f} "
                  f"{r.wall_min:.1f}min")
    LLM = pd.read_csv(RESULTS_CSV).drop_duplicates(["seed","regime"], keep="last")
else:
    p = OUT/"stage2_llm_regimes.csv"
    LLM = pd.read_csv(p).drop_duplicates(["seed","regime"], keep="last") if p.exists() else pd.DataFrame()
    print("stage 2 disabled. Run it only if stage 1 reported a material shift.")

## 4. The gap, and what the data excludes

"No significant difference" is a statement about what was not detected. Reporting the interval
the paired difference occupies converts it into a statement about the effect sizes the data rule
out, which is what is required for a statistically cautious interpretation of the result.

In [ ]:
if not LLM.empty and not RF.empty:
    rows=[]
    for reg in sorted(set(LLM.regime)&set(RF.regime)):
        a = LLM[LLM.regime==reg].set_index("seed")
        b = RF[RF.regime==reg].set_index("seed")
        k = sorted(set(a.index)&set(b.index))
        for met in ("accuracy","macro_f1"):
            d = a.loc[k,met].to_numpy()-b.loc[k,met].to_numpy()
            lo95,hi95 = bci(d); lo90,hi90 = bci(d, conf=0.90)
            rows.append({"regime":reg,"metric":met,"n":len(d),"mean_gap":d.mean(),
                         "ci95_lo":lo95,"ci95_hi":hi95,"p":signflip(d),
                         "p_floor":2.0**-(len(d)-1),
                         "excludes_differences_above":max(abs(lo90),abs(hi90)),
                         "llm_wins":int((d>0).sum())})
    G = save(pd.DataFrame(rows), "gap_llm_vs_anchor.csv")
    print(G.to_string(index=False, float_format=lambda v:f"{v: .4f}"))
    acc = G[G.metric=="accuracy"]
    if len(acc) and acc.mean_gap.abs().max() < 0.05 and (acc.llm_wins < acc.n).all():
        print("\nThe LLM does not surpass the anchor under ANY selector. The parity verdict")
        print("survives development-only feature selection under the paired protocol.")
    print("\nWording for the paper: no advantage was detected at these sample sizes, and the")
    print("data are consistent with any true difference below the 'excludes_differences_above'")
    print("column. This is not equivalence and does not imply production readiness.")
else:
    print("Both stages are needed for a direct LLM comparison; Stage 1 alone quantifies the anchor shift.")

def tex(df, cap, lab):
    (OUT/f"{lab}.tex").write_text(
        "\\begin{table}[htbp]\n\\centering\n\\caption{"+cap+"}\n\\label{t:"+lab+"}\n\\footnotesize\n"
        + df.to_latex(index=False, escape=True, float_format="%.4f") + "\\end{table}\n")
    print(f"[saved] {lab}.tex")

if not RF.empty:
    tex(RF.groupby("regime")[["accuracy","macro_f1"]].mean().reset_index(),
        "Random Forest anchor under three feature-selection regimes on identical in-context and "
        "test rows. The pooled selector reproduces the reported protocol; the other two exclude "
        "test labels.", "axis2-mi-regimes")
if not LLM.empty and not RF.empty:
    tex(G[["regime","metric","mean_gap","ci95_lo","ci95_hi","p","excludes_differences_above"]],
        "Paired difference between the strongest language model and the Random Forest anchor "
        "under each feature-selection regime.", "axis2-mi-gap")

In [ ]:
files = sorted(p for p in OUT.rglob("*") if p.is_file() and p.suffix not in (".zip",".parquet"))
arc = Path(f"axis2_mi_leakage_{datetime.now(timezone.utc):%Y%m%d-%H%M%S}.zip")
with zipfile.ZipFile(arc,"w",zipfile.ZIP_DEFLATED) as z:
    for p in files: z.write(p, arcname=str(Path("axis2_mi")/p.relative_to(OUT)))
print(f"{arc} ({arc.stat().st_size/1e6:.2f} MB, {len(files)} files)")
try:
    from google.colab import files as cf; cf.download(str(arc))
except Exception: print(arc.resolve())